# Temporal Multiplex Directed Networks for the Semiconductor Industry

A Temporal Multiplex Directed Network $\mathcal{M}$ is defined as a sequence of layers $L = \{L_1, L_2, \dots, L_M\}$, where each layer represents a different type of interaction (Financial, Supply Chain, etc.) over time steps $t \in \{1, \dots, T\}$.

The state of the network at any time $t$ is represented by a Supra-Adjacency Tensor $\mathcal{A}$: $$\mathcal{A}_{i,j, \alpha}(t)$$
Where: 
$i, j \in \{1, \dots, N\}$ are the semiconductor companies (nodes). 
$\alpha \in \{1, \dots, M\}$ is the specific layer (e.g., $\alpha=1$ for the Financial Layer, $\alpha=2$ for the Supply Chain Layer, $\alpha=3$ for the Ownership Layer). $t$ is the temporal window (e.g., the specific week).

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px
import matplotlib.pyplot as plt
import networkx as nx
from py_scripts.project2 import financial_layer as fl

ModuleNotFoundError: No module named 'numpy'

## Financial Layer

TODO WRITE A SUMMARY OF THE STEPS DONE AND WHAT WAS ACHIEVED


### DATA ACQUISITION

In [ ]:



foundries = {
    "TSM": "Taiwan Semiconductor Manufacturing Company Limited",
    "SMSN.IL": "Samsung Electronics Co., Ltd.",
    "INTC": "Intel Corporation",
    "UMC": "United Microelectronics Corporation",
    "GFS": "GlobalFoundries Inc.",
    "0981.HK": "Semiconductor Manufacturing International Corporation",
}

fabless_designers = {
    "NVDA": "NVIDIA Corporation",
    "AMD": "Advanced Micro Devices, Inc.",
    "AVGO": "Broadcom Inc.",
    "ARM": "Arm Holdings plc",
    "QCOM": "QUALCOMM Incorporated",
    "2454.TW": "MediaTek Inc.",
    "MRVL": "Marvell Technology, Inc.",
    "ALAB": "Astera Labs, Inc.",
}

memory = {
    "MU": "Micron Technology, Inc.",
    "000660.KS": "SK Hynix Inc.",
}

wfe = {
    "ASML": "ASML Holding N.V.",
    "AMAT": "Applied Materials, Inc.",
    "LRCX": "Lam Research Corporation",
    "KLAC": "KLA Corporation",
    "8035.T": "Tokyo Electron Limited",
    "6857.T": "Advantest Corporation",
    "TER": "Teradyne, Inc.",
    "6920.T": "Lasertec Corporation",
    "SNPS": "Synopsys, Inc.",
    "CDNS": "Cadence Design Systems, Inc.",
}

osat_packaging = {
    "3711.TW": "ASE Technology Holding Co., Ltd.",
    "AMKR": "Amkor Technology, Inc.",
    "042700.KS": "Hanmi Semiconductor Co., Ltd.",
}

analog_auto_power = {
    "TXN": "Texas Instruments Incorporated",
    "ADI": "Analog Devices, Inc.",
    "NXPI": "NXP Semiconductors N.V.",
    "STNE": "STMicroelectronics N.V.",
    "ON": "ON Semiconductor Corporation",
    "IFX.DE": "Infineon Technologies AG",
    "MCHP": "Microchip Technology Incorporated"
}

# Organize spheres
spheres = {
    "Foundries": foundries,
    "Fabless Designers": fabless_designers,
    "Memory": memory,
    "WFE (Equipment)": wfe,
    "OSAT & Packaging": osat_packaging,
    "Analog/Auto/Power": analog_auto_power,
}



In [ ]:
# Download and visualize each sphere with 3h interval
for sphere_name, tickers_dict in spheres.items():
    data = yf.download(list(tickers_dict.keys()), start="2023-01-01", prepost=False, progress=False)['Close']
    
    fig = px.line(data.reset_index(), x='Date', y=data.columns,
                  title=f'{sphere_name} - Close Price (Since 2023, 3h Intervals)',
                  labels={'value': 'Close Price (USD)', 'variable': 'Ticker'})
    fig.update_layout(xaxis_title='Date', yaxis_title='Price (USD)')
    # fig.show()

In [ ]:
# Compute and visualize returns for each sphere
total_returns = pd.DataFrame()
for sphere_name, tickers_dict in spheres.items():
    data = yf.download(list(tickers_dict.keys()), start="2024-01-01", interval='1h', prepost=False, progress=False)['Close']
    returns = np.log(data / data.shift(1)).dropna()
    total_returns = pd.concat([total_returns, returns], axis=1)

    fig = px.line(returns.reset_index(), x='Date', y=returns.columns,
                  title=f'{sphere_name} - Returns (Since 2024, 1h Intervals)',
                  labels={'value': 'Returns', 'variable': 'Ticker'})
    fig.update_layout(xaxis_title='Date', yaxis_title='Returns')
    # fig.show()

NameError: name 'pd' is not defined

### PCA on Returns

The denoising of semiconductor returns relies on the spectral decomposition of the empirical correlation matrix $C$, where the returns are first standardized to unit variance. $$C = \frac{1}{T} Z^T Z = V \Lambda V^T$$ By applying the Marchenko-Pastur theorem, we identify a theoretical noise boundary $\lambda_{max}$ that separates structural market signals from random eigenvalues. $$\lambda_{max} = \sigma^2 (1 + \sqrt{N/T})^2$$  We perform a low-rank reconstruction by projecting the returns $Z$ onto only the $k$ most significant eigenvectors, effectively filtering the data. $$\hat{Z} = (ZV_{sig})V_{sig}^T$$ 
This filtering prevents Sparse VAR Lasso from overfitting to random artifacts, ensuring that the lead-lag edges fed into the GNN represent true structural dependencies rather than coincidental noise.


In [ ]:
window = 35 * 7 
assets_num = 38
all_denoised_windows = []

for i in range(window, len(total_returns), 7):
    
    window_returns = total_returns.iloc[i-window:i]
    
    # Standardize the window returns
    window_mean = window_returns.mean()
    window_std = window_returns.std()
    standardized_slice = (window_returns - window_mean) / window_std
    
    eigenvalues, eigenvectors = fl.PCA(standardized_slice)

    q = window / assets_num
    lambda_max = (1 + np.sqrt(1/q))**2
    print(f"Window {i//7}: λ_max = {lambda_max:.4f}")
    significant_components = np.sum(eigenvalues > lambda_max)
    
    # Project the returns on the eigenvectors
    pca_projections = standardized_slice.values @ eigenvectors
    pca_projections[:, significant_components:] = 0

    denoised_standardized = pca_projections @ eigenvectors.T
    
    # Unstandardize the denoised data
    denoised_final = (denoised_standardized * window_std.values) + window_mean.values
    
    df_denoised = pd.DataFrame(denoised_final, 
                               index=window_returns.index, 
                               columns=window_returns.columns)
    all_denoised_windows.append(df_denoised)

### Breaking Symmetry in the Financial Layer

To transform a standard undirected correlation into a Directed Lead-Lag Network, we define the directed adjacency matrix $A^{(dir)}$ using a time-shifted correlation.

For any two assets $i$ and $j$, the directed edge weight $E_{i \to j}$ is calculated as:$$E_{i \to j}(t) = \text{corr}(R_{i, t}, R_{j, t+1})$$
Conversely, the influence of $j$ on $i$ is:$$E_{j \to i}(t) = \text{corr}(R_{j, t}, R_{i, t+1})$$
In this construction, $A^{(dir)}$ is asymmetric ($E_{i \to j} \neq E_{j \to i}$), representing the directional flow of information from a "leader" to a "lagger."

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Date    0 non-null      object
 1   Graph   0 non-null      object
dtypes: object(2)
memory usage: 132.0+ bytes


,Date,Graph


In [ ]:
# Visualize the last graph for sanity check
last_date, last_graph = graphs_df.iloc[-1]

plt.figure(figsize=(14, 10))
pos = nx.spring_layout(last_graph, k=0.5, iterations=50)
nx.draw_networkx_nodes(last_graph, pos, node_color='lightblue', node_size=500, alpha=0.9)
nx.draw_networkx_labels(last_graph, pos, font_size=9, font_weight='bold')
nx.draw_networkx_edges(last_graph, pos, edge_color='gray', arrows=True, arrowsize=15, 
                       width=0.8, connectionstyle='arc3,rad=0.1', alpha=0.6)
plt.title(f'Directed Lead-Lag Network - {last_date.date()}', fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

IndexError: single positional indexer is out-of-bounds

In [ ]:
#visualize the denoised correlation matrix
plt.figure(figsize=(12, 10))
plt.imshow(C_final_df, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Denoised Correlation')
plt.title('Denoised Correlation Matrix of Semiconductor Stocks')
plt.xticks(ticks=np.arange(len(C_final_df.columns)), labels=C_final_df.columns, rotation=90) 
plt.yticks(ticks=np.arange(len(C_final_df.index)), labels=C_final_df.index)
plt.tight_layout()

NameError: name 'C_final_df' is not defined

<Figure size 1200x1000 with 0 Axes>